# COMP90024 Assignment 2: Data Analysis and Visualization (COMP90024 作業2：數據分析與視覺化)

**Group 57**
- Josh Childs (1549150)
- Enzo Noel (1675346)
- Molly Stow (1356089)
- YU-WEI LIN (1537312)

This notebook aims to address the defined analysis goals by fetching data from the Fission backend and generating relevant visualizations.
(本筆記本旨在通過從 Fission 後端獲取數據並生成相關視覺化來實現定義的分析目標。)

In [ ]:
# Basic Imports and Setup (基本匯入與設定)
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from datetime import datetime, timedelta

# Load functions from frontend_functions.py (從 frontend_functions.py 載入函式)
# Ensure frontend_functions.py is in the same directory or Python path (確保 frontend_functions.py 位於同一目錄或 Python 路徑中)
try:
    from frontend_functions import (
        get_wordcloud_data, # This function fetches entity data from /ui/named-entities/label/{label} (此函式從 /ui/named-entities/label/{label} 獲取實體數據)
        wordcloud_from_data, 
        get_top_each_platform, plot_top_each_platform,
        plot_source_sentiment, # This plots stacked area charts for overall sentiment (此函式繪製整體情緒的堆疊區域圖)
        plot_sentiment_across_platforms, # This plots sentiment for specific keywords/entities (此函式繪製特定關鍵字/實體的情緒圖)
        comparison_plot_keyword_counts, # For comparing keyword counts across platforms on one graph (用於在一個圖表上比較各平台的關鍵字計數)
        plot_keyword_counts, # For plotting keyword counts per platform on separate graphs (用於在獨立圖表上繪製各平台的關鍵字計數)
        labels as platform_display_labels, # Renaming to avoid conflict if 'labels' is used elsewhere (重新命名以避免與其他地方使用的 'labels' 衝突)
        entity_labels,
        fission as FISSION_ROUTER_URL, # Use the fission URL from the imported file (使用匯入檔案中的 fission URL)
        dataframe # Ensure dataframe function is imported if used by plot_stacked_bar_sentiment (確保 dataframe 函式已匯入，供 plot_stacked_bar_sentiment 使用)
    )
    print("Successfully imported from frontend_functions.py (成功從 frontend_functions.py 匯入)")
except ImportError as e:
    print(f"Error importing from frontend_functions.py: {e} (從 frontend_functions.py 匯入時發生錯誤)")
    print("Please ensure frontend_functions.py is in the correct location and all its dependencies are installed. (請確保 frontend_functions.py 位於正確位置且所有依賴均已安裝。)")
    # Define FISSION_ROUTER_URL if import fails, so cells below don't immediately break (如果匯入失敗，定義 FISSION_ROUTER_URL，以免下方儲存格立即出錯)
    FISSION_ROUTER_URL = "http://localhost:9090" 
    # Define dummy dataframe function if import fails for plot_stacked_bar_sentiment (如果匯入失敗，為 plot_stacked_bar_sentiment 定義虛擬 dataframe 函式)
    def dataframe(data, start, end):
        print("[Dummy] dataframe function called. (虛擬 dataframe 函式已呼叫)")
        # Create a minimal DataFrame to prevent errors, assuming data is dict of dicts (建立最小 DataFrame 以防出錯，假設數據是字典的字典)
        try:
            return pd.DataFrame.from_dict(data, orient='index')
        except:
            return pd.DataFrame()
    platform_display_labels = {"openaus": "Open Australia", "bluesky": "Bluesky", "reddit": "Reddit"}
    entity_labels = {"ORG": "Organisations (組織)", "PER": "People (人物)", "LOC": "Locations (地點)", "EVENT": "Events (事件)"}
    # Define a dummy get_wordcloud_data if the import fails so the rest of the notebook can be checked (如果匯入失敗，定義虛擬 get_wordcloud_data，以便檢查筆記本的其餘部分)
    def get_wordcloud_data(label):
        print(f"[Dummy] get_wordcloud_data (for entities) called for label: {label} (虛擬 get_wordcloud_data (用於實體) 已為標籤 {label} 呼叫)")
        return {
            "reddit": {"Dummy Entity R1": 10, "Dummy Entity R2": 8},
            "bluesky": {"Dummy Entity B1": 15, "Dummy Entity B2": 5},
            "openaus": {"Dummy Entity O1": 12, "Dummy Entity O2": 7}
        }
    def get_top_each_platform(data, count=10, normalise=False):
        print("[Dummy] get_top_each_platform called. (虛擬 get_top_each_platform 已呼叫)")
        return {platform: list(entities.items())[:count] for platform, entities in data.items()} if data else {}
    def plot_top_each_platform(data, label, normalised=True):
        print(f"[Dummy] plot_top_each_platform called for {label}. (虛擬 plot_top_each_platform 已為 {label} 呼叫)")
    def wordcloud_from_data(label, data):
        print(f"[Dummy] wordcloud_from_data called for {label}. (虛擬 wordcloud_from_data 已為 {label} 呼叫)")
    def plot_sentiment_across_platforms(keyword_list, keyword_type, results=None, fission_url=None):
        print(f"[Dummy] plot_sentiment_across_platforms called for {keyword_list}. (虛擬 plot_sentiment_across_platforms 已為 {keyword_list} 呼叫)")
        return {}
    def comparison_plot_keyword_counts(platforms, start, keyword, data=None, normalise=True):
        print(f"[Dummy] comparison_plot_keyword_counts called for {keyword}. (虛擬 comparison_plot_keyword_counts 已為 {keyword} 呼叫)")
        return {}

# Load autoreload extension for development convenience (為方便開發載入 autoreload 擴充功能)
%load_ext autoreload
%autoreload 2

# Common parameters (can be adjusted for different analyses) (通用參數 (可為不同分析進行調整))
DEFAULT_START_DATE = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d") # Default to last 30 days (預設為過去30天)
DEFAULT_END_DATE = datetime.now().strftime("%Y-%m-%d")
COMMON_PLATFORMS = ["reddit", "bluesky", "openaus"]

## Analysis Goal 1: What are people talking about? (topics across platforms)
## (分析目標1：大家在談論什麼？(跨平台的話題))

**Tasks (任務):**
- Word cloud for Bluesky, Reddit, and debate topics (OpenAustralia). (為 Bluesky、Reddit 和辯論話題 (OpenAustralia) 製作文字雲。)
- Compare top 10 topics/entities for each platform. (比較各平台的十大話題/實體。)

In [ ]:
# --- Goal 1: Word Clouds and Top Entities (目標1：文字雲和熱門實體) ---
print(f"\n--- Running Analysis Goal 1: Topics and Entities (執行分析目標1：話題和實體) ---")

# Fetch data for word clouds (using ORG as an example entity type) (獲取文字雲數據 (以 ORG 作為實體類型範例))
print("\nFetching data for ORG word clouds... (正在獲取 ORG 文字雲數據...)")
org_wc_data = get_wordcloud_data("ORG") # From frontend_functions.py
if org_wc_data:
    wordcloud_from_data("ORG", org_wc_data) # From frontend_functions.py
else:
    print("Could not fetch ORG data for word clouds. (無法獲取 ORG 文字雲數據。)")

print("\nFetching data for PERSON word clouds... (正在獲取 PERSON 文字雲數據...)")
person_wc_data = get_wordcloud_data("PER") # 'PER' is often used for Person entities (NER 常用 PER 代表人物實體)
if person_wc_data:
    wordcloud_from_data("PER", person_wc_data) 
else:
    print("Could not fetch PERSON data for word clouds. (無法獲取 PERSON 文字雲數據。)")

print("\nFetching data for LOCATION word clouds... (正在獲取 LOCATION 文字雲數據...)")
loc_wc_data = get_wordcloud_data("LOC")
if loc_wc_data:
    wordcloud_from_data("LOC", loc_wc_data)
else:
    print("Could not fetch LOCATION data for word clouds. (無法獲取 LOCATION 文字雲數據。)")

# Get and plot top 10 entities for each platform of each type (獲取並繪製各平台各類型的前10名實體)
entity_types_to_plot = ["ORG", "PER", "LOC"]
for entity_type in entity_types_to_plot:
    print(f"\nProcessing top entities for type: {entity_type} ({entity_labels.get(entity_type, entity_type)}) (正在處理類型為 {entity_type} ({entity_labels.get(entity_type, entity_type)}) 的熱門實體)")
    raw_entity_data = get_wordcloud_data(entity_type) # Use get_wordcloud_data to fetch raw entity data (使用 get_wordcloud_data 獲取原始實體數據)
    if raw_entity_data:
        top_entities = get_top_each_platform(raw_entity_data, count=10, normalise=False) # Get raw counts (獲取原始計數)
        if top_entities:
            plot_top_each_platform(top_entities, entity_labels.get(entity_type, entity_type), normalised=False)
            
            # Create Heatmap for these top entities (using raw counts) (為這些熱門實體建立熱力圖 (使用原始計數))
            try:
                heatmap_df_data = {}
                all_top_entity_names = set()
                for platform_name_internal, platform_entity_list in top_entities.items(): 
                    for entity, _ in platform_entity_list:
                        all_top_entity_names.add(entity)
                
                for platform_name_internal, platform_entity_list in top_entities.items():
                    platform_display_name = platform_display_labels.get(platform_name_internal, platform_name_internal)
                    entity_counts_for_platform = {entity: count for entity, count in platform_entity_list}
                    heatmap_df_data[platform_display_name] = {entity_name: entity_counts_for_platform.get(entity_name, 0) for entity_name in all_top_entity_names}

                if heatmap_df_data and all_top_entity_names:
                    heatmap_df = pd.DataFrame(heatmap_df_data).reindex(list(all_top_entity_names))
                    heatmap_df = heatmap_df.fillna(0).astype(int)
                    heatmap_df['total'] = heatmap_df.sum(axis=1)
                    heatmap_df = heatmap_df.sort_values(by='total', ascending=False).drop(columns='total')
                    heatmap_df = heatmap_df.head(10) 

                    if not heatmap_df.empty:
                        plt.figure(figsize=(12, 8))
                        sns.heatmap(heatmap_df, annot=True, fmt="d", cmap="viridis", linewidths=.5)
                        plt.title(f"Top Mentioned {entity_labels.get(entity_type, entity_type)} Across Platforms (Counts)", fontsize=16)
                        plt.xlabel("Platform", fontsize=12)
                        plt.ylabel(f"{entity_labels.get(entity_type, entity_type)} Entity", fontsize=12)
                        plt.xticks(rotation=45, ha='right')
                        plt.yticks(rotation=0)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print(f"No data to plot heatmap for {entity_type}. (沒有數據可為 {entity_type} 繪製熱力圖。)")
                else:
                    print(f"Could not prepare data for heatmap for {entity_type}. (無法為 {entity_type} 準備熱力圖數據。)")
            except Exception as e:
                print(f"Error generating heatmap for {entity_type}: {e} (為 {entity_type} 生成熱力圖時發生錯誤)")
                import traceback
                traceback.print_exc()
        else:
            print(f"Could not get top entities for type: {entity_type} (無法獲取類型為 {entity_type} 的熱門實體)")
    else:
        print(f"Could not fetch raw entity data for type: {entity_type} (無法獲取類型為 {entity_type} 的原始實體數據)")

## Analysis Goal 2: How do the overall sentiments compare? (sentiment across platforms and time)
## (分析目標2：整體情緒如何比較？(跨平台和時間的情緒))

**Tasks (任務):**
- Sentiment stacked bar chart across time for Bluesky, Reddit, and debates (OpenAustralia).
 (為 Bluesky、Reddit 和辯論 (OpenAustralia) 製作隨時間變化的情緒堆疊長條圖。)

In [ ]:
# --- Goal 2: Overall Sentiment Comparison (Stacked Bar Charts) (目標2：整體情緒比較 (堆疊長條圖)) ---
print(f"\n--- Running Analysis Goal 2: Overall Sentiment Comparison (執行分析目標2：整體情緒比較) ---")

goal2_start_date = DEFAULT_START_DATE
goal2_end_date = DEFAULT_END_DATE

print(f"Fetching overall sentiment data from {goal2_start_date} to {goal2_end_date}... (正在從 {goal2_start_date} 到 {goal2_end_date} 獲取整體情緒數據...)")

raw_overall_sentiment_data = None
try:
    # Fetch data using the generic sentiment endpoint with wildcard keyword
    # (使用帶萬用字元關鍵字的通用情緒端點獲取數據)
    response = requests.get(
        url=f"{FISSION_ROUTER_URL}/ui/sentiment/keyword/*/start/{goal2_start_date}/end/{goal2_end_date}",
        timeout=1500 # Increased timeout for potentially large data (增加超時時間以處理潛在的大量數據)
    )
    response.raise_for_status()
    raw_overall_sentiment_data = response.json()
    print("Successfully fetched overall sentiment data. (成功獲取整體情緒數據。)")
except Exception as e:
    print(f"Error fetching overall sentiment data: {e} (獲取整體情緒數據時發生錯誤)")
    if 'response' in locals() and response is not None:
        print(f"Response text: {response.text} (回應文本)")

def plot_stacked_bar_sentiment(platform_name, daily_sentiment_data, start_str, end_str):
    """Plots daily sentiment as a stacked bar chart (pos, neu, neg proportions). (將每日情緒繪製為堆疊長條圖 (正面、中性、負面比例)。)"""
    if not daily_sentiment_data:
        print(f"No sentiment data for {platform_display_labels.get(platform_name, platform_name)} to plot. (沒有 {platform_display_labels.get(platform_name, platform_name)} 的情緒數據可供繪製。)")
        return
    
    # Use dataframe function from frontend_functions.py to ensure all dates are present (使用 frontend_functions.py 中的 dataframe 函式確保所有日期都存在)
    df = dataframe(daily_sentiment_data, start_str, end_str) 
    df = df.fillna(0) # Fill NaNs for dates with no data (為沒有數據的日期填補 NaN)

    required_cols = ['neg', 'neu', 'pos']
    if not all(col in df.columns for col in required_cols):
        print(f"Data for {platform_display_labels.get(platform_name, platform_name)} is missing required sentiment columns {required_cols} or is empty. (平台 {platform_display_labels.get(platform_name, platform_name)} 的數據缺少必要的情緒欄位 {required_cols} 或為空。)")
        return

    # Calculate proportions (計算比例)
    df_proportions = df[required_cols].copy()
    df_proportions['total_sentiment_values'] = df_proportions[required_cols].sum(axis=1)

    for col in required_cols:
        df_proportions[col] = df_proportions.apply(lambda row: row[col] / row['total_sentiment_values'] if row['total_sentiment_values'] > 0 else (1/3 if col=='neu' else 0), axis=1)
    
    plt.figure(figsize=(15, 6))
    df_proportions[['pos', 'neu', 'neg']].plot(kind='bar', stacked=True, 
                                               color=['forestgreen', 'wheat', 'firebrick'],
                                               width=0.9, ax=plt.gca())
    
    plt.title(f"Daily Sentiment Proportions on {platform_display_labels.get(platform_name, platform_name)}\n({start_str} to {end_str})")
    plt.xlabel("Date (日期)")
    plt.ylabel("Proportion of Sentiments (情緒比例)")
    
    ax = plt.gca()
    tick_positions = np.arange(len(df.index))
    tick_labels = [item.strftime('%Y-%m-%d') for item in df.index]
    
    if len(df.index) > 20: 
        step = max(1, len(df.index) // 10) # Ensure step is at least 1 (確保步長至少為1)
        ax.set_xticks(tick_positions[::step])
        ax.set_xticklabels(tick_labels[::step], rotation=45, ha='right')
    else:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right')

    plt.legend(title='Sentiment (情緒)', loc='upper left', bbox_to_anchor=(1, 1))
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

if raw_overall_sentiment_data:
    for platform in COMMON_PLATFORMS:
        if platform in raw_overall_sentiment_data:
            print(f"\nPlotting stacked bar sentiment for {platform_display_labels.get(platform, platform)}... (正在為 {platform_display_labels.get(platform, platform)} 繪製堆疊長條情緒圖...)")
            plot_stacked_bar_sentiment(platform, raw_overall_sentiment_data[platform], goal2_start_date, goal2_end_date)
        else:
            print(f"No overall sentiment data found for {platform}. (找不到 {platform} 的整體情緒數據。)")
else:
    print("Skipping Goal 2 plots due to data fetching error. (因數據獲取錯誤，跳過目標2的繪圖。)")

## Analysis Goal 3: How do the sentiments of discussions on topics and on (or by) parties or people compare?
## (分析目標3：關於話題、政黨或人物的討論情緒，以及由政黨或人物發表的內容的情緒如何比較？)

**Tasks (任務):**
- Average sentiment over discussions (in debates and online) about key topics (e.g., housing).
 (關於關鍵話題 (例如：住房) 的討論 (辯論和網路上) 的平均情緒。)
- Average sentiment over discussions (in debates and online) about and BY key people and parties.
 (關於關鍵人物和政黨的討論 (辯論和網路上) 以及由他們發表的內容的平均情緒。)

In [ ]:
# --- Goal 3: Sentiment of Discussions on Topics/People/Parties (目標3：關於話題/人物/政黨的討論情緒) ---
print(f"\n--- Running Analysis Goal 3: Sentiment of Discussions (執行分析目標3：討論情緒) ---")

goal3_start_date = DEFAULT_START_DATE
goal3_end_date = DEFAULT_END_DATE

key_topic_goal3 = "housing"
key_people_goal3 = ["Anthony Albanese", "Peter Dutton"] # Example people (範例人物)
key_parties_goal3 = ["Labor Party", "Liberal Party"] # Example parties (ensure these match entities from NER) (範例政黨 (確保這些與 NER 的實體匹配))

print(f"\nFetching sentiment for topic: '{key_topic_goal3}' (正在獲取話題 '{key_topic_goal3}' 的情緒數據)")
# plot_sentiment_across_platforms from frontend_functions.py is suitable here
# It calls /ui/sentiment-averager/type/{keyword_type} with a list of keywords
# (frontend_functions.py 中的 plot_sentiment_across_platforms 在此適用)
# (它使用關鍵字列表呼叫 /ui/sentiment-averager/type/{keyword_type})
topic_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=[key_topic_goal3],
    keyword_type="topic" # Assuming your backend /ui/sentiment-averager can handle 'topic' type (假設您的後端 /ui/sentiment-averager 可以處理 'topic' 類型)
)

print(f"\nFetching sentiment for people: {key_people_goal3} (正在獲取人物 {key_people_goal3} 的情緒數據)")
people_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=key_people_goal3,
    keyword_type="people" # Corresponds to 'PER' for NER, or how your backend handles it (對應 NER 的 'PER'，或您的後端如何處理它)
)

print(f"\nFetching sentiment for parties: {key_parties_goal3} (正在獲取政黨 {key_parties_goal3} 的情緒數據)")
parties_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=key_parties_goal3,
    keyword_type="parties" # Corresponds to 'ORG' for NER, or how your backend handles it (對應 NER 的 'ORG'，或您的後端如何處理它)
)

print("\nGoal 3 Note: The plots above show sentiment of discussions ABOUT the keywords. (目標3注意：以上圖表顯示的是關於這些關鍵字的討論情緒。)")
print("Analyzing sentiment OF speeches BY specific people/parties would require a dedicated backend function and data source (e.g., OpenAustralia transcripts filtered by speaker). (分析特定人物/政黨發表的演講的情緒，需要專用的後端函數和數據源 (例如，按演講者篩選的 OpenAustralia 文字記錄)。)")

## Analysis Goal 4: Do people discuss the recent topics of debates?
## (分析目標4：人們是否討論最近的辯論話題？)

**Tasks (任務):**
- Choose a topic (e.g., housing). (選擇一個話題 (例如：住房)。)
- Graph over time Bluesky/Reddit posts with that topic AND debates with that topic.
 (繪製 Bluesky/Reddit 上包含該話題的貼文以及辯論中包含該話題的內容隨時間變化的圖表。)
- See if discussions rise immediately after (maybe start from a specific date of a debate with a clear topic?).
 (觀察討論是否在辯論之後立即增加 (或許可以從一個有明確主題的辯論的特定日期開始？))

In [ ]:
# --- Goal 4: Discussion of Debate Topics (目標4：辯論話題的討論) ---
print(f"\n--- Running Analysis Goal 4: Discussion of Debate Topics (執行分析目標4：辯論話題的討論) ---")

goal4_topic = "housing"
goal4_start_date = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d") # Look at last 90 days (觀察過去90天)
goal4_end_date = DEFAULT_END_DATE

print(f"Plotting counts for topic '{goal4_topic}' from {goal4_start_date} to {goal4_end_date} across platforms. (正在繪製話題 '{goal4_topic}' 從 {goal4_start_date} 到 {goal4_end_date} 在各平台的計數圖。)")
# comparison_plot_keyword_counts from frontend_functions.py is suitable here (frontend_functions.py 中的 comparison_plot_keyword_counts 在此適用)
counts_data = comparison_plot_keyword_counts(
    platforms=COMMON_PLATFORMS, 
    start=goal4_start_date, 
    keyword=goal4_topic,
    normalise=True # Normalize to compare trends, or False for raw counts (正規化以比較趨勢，或使用 False 顯示原始計數)
)

if counts_data:
    print(f"Successfully plotted counts for topic '{goal4_topic}'. (成功繪製話題 '{goal4_topic}' 的計數圖。)")
else:
    print(f"Could not fetch or plot counts for topic '{goal4_topic}'. (無法獲取或繪製話題 '{goal4_topic}' 的計數圖。)")

print("\nGoal 4 Note: To see if discussions rise immediately after a specific debate, you would: (目標4注意：要觀察特定辯論後討論是否立即增加，您需要：)")
print("1. Identify a specific date of a debate on the chosen topic. (1. 確定所選話題的特定辯論日期。)")
print("2. Adjust the start/end dates for the plot to focus around that debate date. (2. 調整繪圖的開始/結束日期，使其圍繞該辯論日期。)")
print("3. Observe the 'OpenAustralia' count line (representing debates) and compare with Reddit/Bluesky lines. (3. 觀察 'OpenAustralia' 的計數線 (代表辯論) 並與 Reddit/Bluesky 的線條進行比較。)")

--- End of Analysis Notebook (分析筆記本結束) ---